# 03. スキーマ進化（Spark）

Iceberg では、列の追加・名前変更・型の拡張・削除を **データファイルを書き直さずに** 行えます。
各列は名前ではなく内部の **列 ID** で管理されているため、名前を変えても古いファイルの列と正しく対応が取れます。

題材は NYC タクシーのデータです。2025-01 から `cbd_congestion_fee`（マンハッタン中心部の渋滞料金）列が追加されました。
2024-12 のデータでテーブルを作り、後から来た 2025-01 のデータに合わせてスキーマを変えていきます。

- テーブル: `handson.taxi_schema_spark`
- 事前に `make data` でデータを取得しておく

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("03_schema_evolution").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. 元データのスキーマを比べる

2025-01 にだけ `cbd_congestion_fee` 列があります。

In [ ]:
dec = spark.read.parquet("/workspace/data/yellow_tripdata_2024-12.parquet")
jan = spark.read.parquet("/workspace/data/yellow_tripdata_2025-01.parquet")
print("2025-01 にだけある列:", set(jan.columns) - set(dec.columns))

## 2. 2024-12 のデータでテーブルを作る

ハンズオンで扱いやすいよう、一部の列だけを使います。

In [ ]:
sql("DROP TABLE IF EXISTS handson.taxi_schema_spark PURGE")
sql("""
CREATE TABLE handson.taxi_schema_spark (
    vendor_id     INT,
    pickup_at     TIMESTAMP_NTZ,
    trip_distance DOUBLE,
    fare_amount   DOUBLE,
    payment_type  BIGINT,
    total_amount  DOUBLE
) USING iceberg
""")

dec.createOrReplaceTempView("dec")
sql("""
INSERT INTO handson.taxi_schema_spark
SELECT VendorID, tpep_pickup_datetime, trip_distance, fare_amount, payment_type, total_amount FROM dec
""")
sql("SELECT count(*) AS rows FROM handson.taxi_schema_spark")

データファイルの数を覚えておきます。この後スキーマをいくら変えても、既存のファイルは増えも書き換えられもしません。

In [ ]:
def show_files():
    sql("SELECT count(*) AS data_files, sum(record_count) AS rows FROM handson.taxi_schema_spark.files")

show_files()

## 3. 列を追加する

`ADD COLUMN` はメタデータの変更だけで終わります。既存の行の新しい列は NULL として読まれます。

In [ ]:
sql("ALTER TABLE handson.taxi_schema_spark ADD COLUMN cbd_congestion_fee DOUBLE")

jan.createOrReplaceTempView("jan")
sql("""
INSERT INTO handson.taxi_schema_spark
SELECT VendorID, tpep_pickup_datetime, trip_distance, fare_amount, payment_type, total_amount, cbd_congestion_fee FROM jan
""")

sql("""
SELECT date_format(pickup_at, 'yyyy-MM') AS month,
       count(*) AS trips,
       count(cbd_congestion_fee) AS with_cbd_fee,
       round(avg(cbd_congestion_fee), 3) AS avg_cbd_fee
FROM handson.taxi_schema_spark
WHERE pickup_at >= '2024-12-01' AND pickup_at < '2025-02-01'
GROUP BY 1 ORDER BY 1
""")
show_files()

2024-12 の行は、ほぼすべて `cbd_congestion_fee` が NULL です。ファイルは 2025-01 の分だけ増えています。

（月は乗車日時で分けているので、2025-01 のファイルに含まれる「12月31日に乗って1月に降りた」ような行が 2024-12 側に少し数えられています。実データにはこうした揺れがつきものです。）

## 4. 列の名前を変える

`fare_amount` を `fare` に変えます。古いファイルの中の列名は `fare_amount` のままですが、列 ID で対応が取れるので問題なく読めます。

In [ ]:
sql("ALTER TABLE handson.taxi_schema_spark RENAME COLUMN fare_amount TO fare")
sql("SELECT pickup_at, fare FROM handson.taxi_schema_spark WHERE pickup_at >= '2024-12-01' ORDER BY pickup_at LIMIT 3")
show_files()

## 5. 型を広げる

`vendor_id` を INT から BIGINT に広げます。
Iceberg が許すのは、既存のデータをそのまま読める「安全な」型変更だけです（INT → BIGINT、FLOAT → DOUBLE、DECIMAL の精度を上げる など）。

In [ ]:
sql("ALTER TABLE handson.taxi_schema_spark ALTER COLUMN vendor_id TYPE BIGINT")
sql("DESCRIBE TABLE handson.taxi_schema_spark")

逆向き（BIGINT → INT）のような、値が収まらない可能性のある変更はエラーになります。

In [ ]:
try:
    sql("ALTER TABLE handson.taxi_schema_spark ALTER COLUMN vendor_id TYPE INT")
except Exception as e:
    print(type(e).__name__, str(e).splitlines()[0])

## 6. 列を削除して、同じ名前で追加し直す

`payment_type` を削除してから、同じ名前で追加し直します。
新しい列には **新しい列 ID** が振られるので、古いファイルに残っている `payment_type` の値は復活せず、すべて NULL になります。
名前で列を管理している形式（素の Parquet や Hive のテーブル）では、古い値が誤って読まれてしまうことがあります。

In [ ]:
sql("ALTER TABLE handson.taxi_schema_spark DROP COLUMN payment_type")
sql("ALTER TABLE handson.taxi_schema_spark ADD COLUMN payment_type BIGINT")
sql("SELECT count(*) AS rows, count(payment_type) AS non_null_payment_type FROM handson.taxi_schema_spark")
show_files()

## まとめ

- 列の追加・名前変更・型の拡張・削除は、どれもメタデータの変更だけで終わり、データファイルは書き直されない
- 列は名前ではなく列 ID で管理されているので、名前を変えても、削除して同じ名前で追加し直しても、正しく扱われる
- 型の変更は、既存のデータを安全に読める方向（拡張）だけが許される